Objective Build an end-to-end Databricks pipeline using:

PySpark DataFrames

Spark SQL

Delta Lake

CRUD

MERGE / UPSERT

History

Time Travel

VACUUM

Parquet to Delta

Incremental Load

DLT

Unity


Catalog

Governance

In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Retail Supply Chain Analytics").getOrCreate()

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

Part 0 — Prepare Bigger Dataset Students can run this first in a normal Python notebook.

In [3]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]
products_columns = [
"product_id",
"product_name",
"category",
"warehouse_city",
"price",
"stock_quantity"
]
products_df = spark.createDataFrame(products_data, products_columns)

In [4]:
suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]
suppliers_columns = [
"supplier_id",
"supplier_name",
"supplier_city",
"specialization"
]
suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

In [5]:
orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]
orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [6]:
payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]
payments_columns = [
"payment_id",
"order_id",
"bill_amount",
"payment_mode",
"payment_status"
]
payments_df = spark.createDataFrame(payments_data, payments_columns)

Assessment Tasks Part 1 — DataFrame Fundamentals . Display all DataFrames.

1. Display all DataFrames.


In [7]:
products_df.show()
suppliers_df.show()
orders_df.show()
payments_df.show()

+----------+---------------+---------------+--------------+-----+--------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|
+----------+---------------+---------------+--------------+-----+--------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|
|       108|     Toothpaste|  Personal Care|     Ahmedabad|   90|           250|
|       109|       Notebook|     Stationery|     Hyderabad|   75|           500|
|       110|       Pen Pack|

2. Print schema for each DataFrame.


In [8]:
products_df.printSchema()
suppliers_df.printSchema()
orders_df.printSchema()
payments_df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_city: string (nullable = true)
 |-- price: long (nullable = true)
 |-- stock_quantity: long (nullable = true)

root
 |-- supplier_id: long (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_city: string (nullable = true)
 |-- specialization: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- supplier_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



3. Count total records in each DataFrame.



In [9]:
products_df.count()
suppliers_df.count()
orders_df.count()
payments_df.count()

20

4. Display first 10 products.


In [10]:
products_df.show(10)

+----------+-------------+-------------+--------------+-----+--------------+
|product_id| product_name|     category|warehouse_city|price|stock_quantity|
+----------+-------------+-------------+--------------+-----+--------------+
|       101|     Rice Bag|    Groceries|     Hyderabad| 1200|            50|
|       102|  Wheat Flour|    Groceries|     Bengaluru|  900|            80|
|       103|Sunflower Oil|    Groceries|        Mumbai| 1800|            40|
|       104|    Milk Pack|        Dairy|       Chennai|   60|           200|
|       105| Cheese Block|        Dairy|         Delhi|  450|            70|
|       106|         Soap|Personal Care|       Kolkata|  120|           300|
|       107|      Shampoo|Personal Care|          Pune|  320|           150|
|       108|   Toothpaste|Personal Care|     Ahmedabad|   90|           250|
|       109|     Notebook|   Stationery|     Hyderabad|   75|           500|
|       110|     Pen Pack|   Stationery|        Mumbai|  110|           400|

5. Display product name, category, and stock quantity.


In [11]:
products_df.select(
    "product_name",
    "category",
    "stock_quantity"
).show()

+---------------+---------------+--------------+
|   product_name|       category|stock_quantity|
+---------------+---------------+--------------+
|       Rice Bag|      Groceries|            50|
|    Wheat Flour|      Groceries|            80|
|  Sunflower Oil|      Groceries|            40|
|      Milk Pack|          Dairy|           200|
|   Cheese Block|          Dairy|            70|
|           Soap|  Personal Care|           300|
|        Shampoo|  Personal Care|           150|
|     Toothpaste|  Personal Care|           250|
|       Notebook|     Stationery|           500|
|       Pen Pack|     Stationery|           400|
|         LED TV|    Electronics|            15|
|   Refrigerator|    Electronics|            10|
|Washing Machine|    Electronics|            12|
|   Mobile Phone|    Electronics|            35|
|         Laptop|    Electronics|            18|
|Air Conditioner|    Electronics|             9|
|  Mixer Grinder|Home Appliances|            45|
| Water Purifier|Hom

6. Display suppliers from Hyderabad and Chennai.


In [12]:
suppliers_df.filter(
    col("supplier_city").isin("Hyderabad", "Chennai")
).show()

+-----------+----------------+-------------+---------------+
|supplier_id|   supplier_name|supplier_city| specialization|
+-----------+----------------+-------------+---------------+
|        201|   Reddy Traders|    Hyderabad|      Groceries|
|        202| Fresh Dairy Ltd|      Chennai|          Dairy|
|        209|Daily Essentials|    Hyderabad|  Personal Care|
|        210|   Kitchen World|      Chennai|Home Appliances|
+-----------+----------------+-------------+---------------+



7. Display orders with status Delivered.


In [13]:
orders_df.filter(
    col("order_status") == "Delivered"
).show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|
|     302|       102|        201|2024-04-01|      35|   Delivered|
|     303|       111|        204|2024-04-02|       2|   Delivered|
|     305|       115|        204|2024-04-03|       3|   Delivered|
|     306|       104|        202|2024-04-03|      50|   Delivered|
|     308|       117|        206|2024-04-04|       7|   Delivered|
|     310|       119|        206|2024-04-05|      12|   Delivered|
|     311|       120|        210|2024-04-06|       6|   Delivered|
|     312|       113|        204|2024-04-06|       4|   Delivered|
|     314|       109|        205|2024-04-07|      80|   Delivered|
|     315|       110|        205|2024-04-08|     120|   Delivered|
|     317|       107|        209|2024-04-09|      25|   Delive

8. Display pending orders.


In [14]:
orders_df.filter(
    col("order_status") == "Pending"
).show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     304|       114|        208|2024-04-02|       5|     Pending|
|     309|       118|        210|2024-04-05|       4|     Pending|
|     313|       116|        208|2024-04-07|       2|     Pending|
|     319|       112|        208|2024-04-10|       2|     Pending|
+--------+----------+-----------+----------+--------+------------+



9. Display electronics products only.


In [15]:
products_df.filter(
    col("category") == "Electronics"
).show()

+----------+---------------+-----------+--------------+-----+--------------+
|product_id|   product_name|   category|warehouse_city|price|stock_quantity|
+----------+---------------+-----------+--------------+-----+--------------+
|       111|         LED TV|Electronics|         Delhi|45000|            15|
|       112|   Refrigerator|Electronics|       Chennai|38000|            10|
|       113|Washing Machine|Electronics|     Bengaluru|29000|            12|
|       114|   Mobile Phone|Electronics|     Hyderabad|25000|            35|
|       115|         Laptop|Electronics|          Pune|62000|            18|
|       116|Air Conditioner|Electronics|        Mumbai|42000|             9|
+----------+---------------+-----------+--------------+-----+--------------+



10. Display products with stock quantity below 20.

In [16]:
products_df.filter(
    col("stock_quantity") < 20
).show()

+----------+---------------+-----------+--------------+-----+--------------+
|product_id|   product_name|   category|warehouse_city|price|stock_quantity|
+----------+---------------+-----------+--------------+-----+--------------+
|       111|         LED TV|Electronics|         Delhi|45000|            15|
|       112|   Refrigerator|Electronics|       Chennai|38000|            10|
|       113|Washing Machine|Electronics|     Bengaluru|29000|            12|
|       115|         Laptop|Electronics|          Pune|62000|            18|
|       116|Air Conditioner|Electronics|        Mumbai|42000|             9|
+----------+---------------+-----------+--------------+-----+--------------+



Part 2 — DataFrame Transformations

11. Convert order_date into date type.


In [17]:
orders_df = orders_df.withColumn(
    "order_date",
    to_date(col("order_date"))
)
orders_df.show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|
|     302|       102|        201|2024-04-01|      35|   Delivered|
|     303|       111|        204|2024-04-02|       2|   Delivered|
|     304|       114|        208|2024-04-02|       5|     Pending|
|     305|       115|        204|2024-04-03|       3|   Delivered|
|     306|       104|        202|2024-04-03|      50|   Delivered|
|     307|       105|        202|2024-04-04|      18|   Cancelled|
|     308|       117|        206|2024-04-04|       7|   Delivered|
|     309|       118|        210|2024-04-05|       4|     Pending|
|     310|       119|        206|2024-04-05|      12|   Delivered|
|     311|       120|        210|2024-04-06|       6|   Delivered|
|     312|       113|        204|2024-04-06|       4|   Delive

12. Add total_order_value column.


In [18]:
orders_products_df = orders_df.join(
    products_df,
    "product_id"
)

orders_products_df = orders_products_df.withColumn(
    "total_order_value",
    col("quantity") * col("price")
)
orders_products_df.show()

+----------+--------+-----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+-----------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|   product_name|       category|warehouse_city|price|stock_quantity|total_order_value|
+----------+--------+-----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+-----------------+
|       101|     301|        201|2024-04-01|      20|   Delivered|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|            24000|
|       101|     320|        207|2024-04-10|      15|   Delivered|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|            18000|
|       102|     302|        201|2024-04-01|      35|   Delivered|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|            31500|
|       104|     306|        202|2024-04-03|      50|   Delivered|      Milk Pack|

13. Create stock_status column.


In [19]:
products_df = products_df.withColumn(
    "stock_status",
    when(col("stock_quantity") < 20, "Low Stock")
    .otherwise("Available")
)
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|
+----------+---------------+---------------+--------------+-----+--------------+------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|   Available|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|   Available|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|   Available|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|   Available|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|   Available|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|   Available|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|   Available|
|       108|     Toothpaste|  Personal Care|     Ahmedabad| 

14. Create order_priority column.


In [20]:
orders_df = orders_df.withColumn(
    "order_priority",
    when(col("quantity") >= 50, "High")
    .when(col("quantity") >= 20, "Medium")
    .otherwise("Low")
)
orders_df.show()

+--------+----------+-----------+----------+--------+------------+--------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|order_priority|
+--------+----------+-----------+----------+--------+------------+--------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|        Medium|
|     302|       102|        201|2024-04-01|      35|   Delivered|        Medium|
|     303|       111|        204|2024-04-02|       2|   Delivered|           Low|
|     304|       114|        208|2024-04-02|       5|     Pending|           Low|
|     305|       115|        204|2024-04-03|       3|   Delivered|           Low|
|     306|       104|        202|2024-04-03|      50|   Delivered|          High|
|     307|       105|        202|2024-04-04|      18|   Cancelled|           Low|
|     308|       117|        206|2024-04-04|       7|   Delivered|           Low|
|     309|       118|        210|2024-04-05|       4|     Pending|           Low|
|     310|      

15. Create expensive_product_flag column.


In [21]:
products_df = products_df.withColumn(
    "expensive_product_flag",
    when(col("price") > 10000, "Yes")
    .otherwise("No")
)
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|expensive_product_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|   Available|                    No|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|   Available|                    No|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|   Available|                    No|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|   Available|                    No|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|   Available|                    No|
|       106|           Soap|  Personal Care|       Kolkata|  120

16. Convert category names into uppercase.


In [22]:
products_df = products_df.withColumn(
    "category",
    upper(col("category"))
)
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|expensive_product_flag|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|   Available|                    No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|   Available|                    No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|   Available|                    No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|   Available|                    No|
|       105|   Cheese Block|          DAIRY|         Delhi|  450|            70|   Available|                    No|
|       106|           Soap|  PERSONAL CARE|       Kolkata|  120

17. Rename warehouse_city to inventory_city.


In [23]:
products_df = products_df.withColumnRenamed(
    "warehouse_city",
    "inventory_city"
)

18. Create inventory_value column.


In [24]:
products_df = products_df.withColumn(
    "inventory_value",
    col("price") * col("stock_quantity")
)
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|stock_status|expensive_product_flag|inventory_value|
+----------+---------------+---------------+--------------+-----+--------------+------------+----------------------+---------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|   Available|                    No|          60000|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|   Available|                    No|          72000|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|   Available|                    No|          72000|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|   Available|                    No|          12000|
|       105|   Cheese Block|          DAIRY|         Delhi|  450|    

19. Drop temporary columns.


In [25]:
products_df = products_df.drop("stock_status")

20. Create low_stock_flag column.

In [26]:
products_df = products_df.withColumn(
    "low_stock_flag",
    when(col("stock_quantity") < 20, "Yes")
    .otherwise("No")
)
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|                    No|          60000|            No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|                    No|          72000|            No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|                    No|          72000|            No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|                    No|          12000|            No|
|       105|   Cheese Block|          DAIRY|         De

Part 3 — Joins

21. Join products with orders.



In [27]:
products_orders_df = products_df.join(
    orders_df,
    "product_id"
)

22. Join orders with suppliers.


In [28]:
orders_suppliers_df = orders_df.join(
    suppliers_df,
    "supplier_id"
)

23. Join orders with payments.


In [29]:
orders_payments_df = orders_df.join(
    payments_df,
    "order_id"
)

24. Create final joined DataFrame.


In [30]:
final_df = orders_df \
    .join(products_df, "product_id") \
    .join(suppliers_df, "supplier_id") \
    .join(payments_df, "order_id")

25. Display product name, supplier name, quantity, bill amount.


In [31]:
final_df.select(
    "product_name",
    "supplier_name",
    "quantity",
    "bill_amount"
).show()

+---------------+------------------+--------+-----------+
|   product_name|     supplier_name|quantity|bill_amount|
+---------------+------------------+--------+-----------+
|       Rice Bag|     Reddy Traders|      20|      24000|
|    Wheat Flour|     Reddy Traders|      35|      31500|
|         LED TV| Elite Electronics|       2|      90000|
|   Mobile Phone| Smart Electronics|       5|     125000|
|         Laptop| Elite Electronics|       3|     186000|
|      Milk Pack|   Fresh Dairy Ltd|      50|       3000|
|   Cheese Block|   Fresh Dairy Ltd|      18|       8100|
|  Mixer Grinder| HomeNeeds Pvt Ltd|       7|      24500|
| Water Purifier|     Kitchen World|       4|      48000|
|    Ceiling Fan| HomeNeeds Pvt Ltd|      12|      33600|
|      Gas Stove|     Kitchen World|       6|      33000|
|Washing Machine| Elite Electronics|       4|     116000|
|Air Conditioner| Smart Electronics|       2|      84000|
|       Notebook|        OfficeKart|      80|       6000|
|       Pen Pa

26. Find suppliers serving different cities.


In [32]:
suppliers_df.select(
    "supplier_name",
    "supplier_city"
).distinct().show()

+------------------+-------------+
|     supplier_name|supplier_city|
+------------------+-------------+
|     Reddy Traders|    Hyderabad|
|CarePlus Suppliers|       Mumbai|
|   Fresh Dairy Ltd|      Chennai|
| Elite Electronics|        Delhi|
|        OfficeKart|    Bengaluru|
|  Daily Essentials|    Hyderabad|
| Smart Electronics|      Kolkata|
|  National Grocers|    Ahmedabad|
| HomeNeeds Pvt Ltd|         Pune|
|     Kitchen World|      Chennai|
+------------------+-------------+



27. Find delivered orders with paid payments.


In [33]:
final_df.filter(
    (col("order_status") == "Delivered") &
    (col("payment_status") == "Paid")
).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     301|        201|       101|2024-04-01|      20|   Delivered|        Medium|    

28. Find pending orders with pending payments.


In [34]:
final_df.filter(
    (col("order_status") == "Pending") &
    (col("payment_status") == "Pending")
).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+
|     304|        208|       114|2024-04-02|       5|     Pending|           Low|   Mobi

29. Find cancelled orders with cancelled payments.


In [35]:
final_df.filter(
    (col("order_status") == "Cancelled") &
    (col("payment_status") == "Cancelled")
).show()

+--------+-----------+----------+----------+--------+------------+--------------+------------+-------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+--------------+----------+-----------+------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|product_name|     category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city|specialization|payment_id|bill_amount|payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+------------+-------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+--------------+----------+-----------+------------+--------------+
|     307|        202|       105|2024-04-04|      18|   Cancelled|           Low|Cheese Block|        DAIR

30. Find products ordered multiple times.

In [36]:
final_df.groupBy("product_name") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+------------+-----+
|product_name|count|
+------------+-----+
|    Rice Bag|    2|
+------------+-----+



Part 4 — Aggregations

31. Count products by category.



In [37]:
products_df.groupBy("category").count().show()

+---------------+-----+
|       category|count|
+---------------+-----+
|      GROCERIES|    3|
|          DAIRY|    2|
|     STATIONERY|    2|
|  PERSONAL CARE|    3|
|    ELECTRONICS|    6|
|HOME APPLIANCES|    4|
+---------------+-----+



32. Count orders by status.


In [38]:
orders_df.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Cancelled|    2|
|   Delivered|   14|
|     Pending|    4|
+------------+-----+



33. Count suppliers by city.


In [39]:
suppliers_df.groupBy("supplier_city").count().show()

+-------------+-----+
|supplier_city|count|
+-------------+-----+
|      Chennai|    2|
|       Mumbai|    1|
|        Delhi|    1|
|    Bengaluru|    1|
|    Hyderabad|    2|
|    Ahmedabad|    1|
|      Kolkata|    1|
|         Pune|    1|
+-------------+-----+



34. Calculate total revenue.


In [40]:
final_df.agg(
    sum("bill_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|       938700|
+-------------+



35. Calculate average bill amount.


In [41]:
final_df.agg(
    avg("bill_amount").alias("average_bill")
).show()

+------------+
|average_bill|
+------------+
|     46935.0|
+------------+



36. Calculate total revenue by category.


In [42]:
final_df.groupBy("category") \
    .agg(sum("bill_amount").alias("revenue")) \
    .show()

+---------------+-------+
|       category|revenue|
+---------------+-------+
|      GROCERIES|  73500|
|          DAIRY|  11100|
|    ELECTRONICS| 677000|
|     STATIONERY|  19200|
|  PERSONAL CARE|  18800|
|HOME APPLIANCES| 139100|
+---------------+-------+



37. Calculate total revenue by supplier.


In [43]:
final_df.groupBy("supplier_name") \
    .agg(sum("bill_amount").alias("revenue")) \
    .show()

+------------------+-------+
|     supplier_name|revenue|
+------------------+-------+
|     Reddy Traders|  55500|
| HomeNeeds Pvt Ltd|  58100|
|   Fresh Dairy Ltd|  11100|
|     Kitchen World|  81000|
| Elite Electronics| 392000|
|  National Grocers|  18000|
|        OfficeKart|  19200|
| Smart Electronics| 285000|
|CarePlus Suppliers|  10800|
|  Daily Essentials|   8000|
+------------------+-------+



38. Calculate total revenue by city.


In [44]:
final_df.groupBy("supplier_city") \
    .agg(sum("bill_amount").alias("revenue")) \
    .show()

+-------------+-------+
|supplier_city|revenue|
+-------------+-------+
|      Chennai|  92100|
|       Mumbai|  10800|
|    Ahmedabad|  18000|
|      Kolkata| 285000|
|         Pune|  58100|
|        Delhi| 392000|
|    Bengaluru|  19200|
|    Hyderabad|  63500|
+-------------+-------+



39. Find top-selling products.


In [45]:
final_df.groupBy("product_name") \
    .agg(sum("quantity").alias("total_quantity")) \
    .orderBy(desc("total_quantity")) \
    .show()

+---------------+--------------+
|   product_name|total_quantity|
+---------------+--------------+
|       Pen Pack|           120|
|       Notebook|            80|
|           Soap|            60|
|      Milk Pack|            50|
|     Toothpaste|            40|
|       Rice Bag|            35|
|    Wheat Flour|            35|
|        Shampoo|            25|
|   Cheese Block|            18|
|    Ceiling Fan|            12|
|  Mixer Grinder|             7|
|      Gas Stove|             6|
|   Mobile Phone|             5|
|Washing Machine|             4|
| Water Purifier|             4|
|         Laptop|             3|
|   Refrigerator|             2|
|Air Conditioner|             2|
|         LED TV|             2|
+---------------+--------------+



40. Find lowest-selling products.

In [46]:
final_df.groupBy("product_name") \
    .agg(sum("quantity").alias("total_quantity")) \
    .orderBy("total_quantity") \
    .show()

+---------------+--------------+
|   product_name|total_quantity|
+---------------+--------------+
|   Refrigerator|             2|
|Air Conditioner|             2|
|         LED TV|             2|
|         Laptop|             3|
|Washing Machine|             4|
| Water Purifier|             4|
|   Mobile Phone|             5|
|      Gas Stove|             6|
|  Mixer Grinder|             7|
|    Ceiling Fan|            12|
|   Cheese Block|            18|
|        Shampoo|            25|
|       Rice Bag|            35|
|    Wheat Flour|            35|
|     Toothpaste|            40|
|      Milk Pack|            50|
|           Soap|            60|
|       Notebook|            80|
|       Pen Pack|           120|
+---------------+--------------+



Part 5 — Spark SQL

41. Create temp views for all DataFrames.


In [47]:
products_df.createOrReplaceTempView("products")
suppliers_df.createOrReplaceTempView("suppliers")
orders_df.createOrReplaceTempView("orders")
payments_df.createOrReplaceTempView("payments")
final_df.createOrReplaceTempView("final_table")

42. Show all products using SQL.


In [48]:
spark.sql("""
SELECT * FROM products
""").show()

+----------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|                    No|          60000|            No|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|                    No|          72000|            No|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|                    No|          72000|            No|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|                    No|          12000|            No|
|       105|   Cheese Block|          DAIRY|         De

43. Find electronics orders using SQL.


In [49]:
spark.sql("""
SELECT *
FROM final_table
WHERE category = 'ELECTRONICS'
""").show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+-----------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+--------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|   category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city|specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+-----------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+--------------+----------+-----------+-------------+--------------+
|     303|        204|       111|2024-04-02|       2|   Delivered|           Low|         LED TV|ELECTR

44. Find revenue by category.


In [50]:
spark.sql("""
SELECT category,
SUM(bill_amount) AS revenue
FROM final_table
GROUP BY category
""").show()

+---------------+-------+
|       category|revenue|
+---------------+-------+
|      GROCERIES|  73500|
|          DAIRY|  11100|
|    ELECTRONICS| 677000|
|     STATIONERY|  19200|
|  PERSONAL CARE|  18800|
|HOME APPLIANCES| 139100|
+---------------+-------+



45. Find revenue by supplier.


In [51]:
spark.sql("""
SELECT supplier_name,
SUM(bill_amount) AS revenue
FROM final_table
GROUP BY supplier_name
""").show()

+------------------+-------+
|     supplier_name|revenue|
+------------------+-------+
|     Reddy Traders|  55500|
| HomeNeeds Pvt Ltd|  58100|
|   Fresh Dairy Ltd|  11100|
|     Kitchen World|  81000|
| Elite Electronics| 392000|
|  National Grocers|  18000|
|        OfficeKart|  19200|
| Smart Electronics| 285000|
|CarePlus Suppliers|  10800|
|  Daily Essentials|   8000|
+------------------+-------+



46. Find top 5 highest bill orders.


In [52]:
spark.sql("""
SELECT *
FROM final_table
ORDER BY bill_amount DESC
LIMIT 5
""").show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+-----------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+--------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|   category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city|specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+-----------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+--------------+----------+-----------+-------------+--------------+
|     305|        204|       115|2024-04-03|       3|   Delivered|           Low|         Laptop|ELECTR

47. Count orders per supplier.


In [53]:
spark.sql("""
SELECT supplier_name,
COUNT(order_id) AS total_orders
FROM final_table
GROUP BY supplier_name
""").show()

+------------------+------------+
|     supplier_name|total_orders|
+------------------+------------+
|     Reddy Traders|           2|
| HomeNeeds Pvt Ltd|           2|
|   Fresh Dairy Ltd|           2|
|     Kitchen World|           2|
| Elite Electronics|           3|
|  National Grocers|           1|
|        OfficeKart|           2|
| Smart Electronics|           3|
|CarePlus Suppliers|           2|
|  Daily Essentials|           1|
+------------------+------------+



48. Count orders per category.


In [54]:
spark.sql("""
SELECT category,
COUNT(order_id) AS total_orders
FROM final_table
GROUP BY category
""").show()

+---------------+------------+
|       category|total_orders|
+---------------+------------+
|      GROCERIES|           3|
|          DAIRY|           2|
|    ELECTRONICS|           6|
|     STATIONERY|           2|
|  PERSONAL CARE|           3|
|HOME APPLIANCES|           4|
+---------------+------------+



49. Find average payment per mode.


In [55]:
spark.sql("""
SELECT payment_mode,
AVG(bill_amount) AS avg_payment
FROM final_table
GROUP BY payment_mode
""").show()

+-------------+-----------+
| payment_mode|avg_payment|
+-------------+-----------+
|  Credit Card|    32250.0|
|Bank Transfer|   117000.0|
|         Cash|    12450.0|
|   Debit Card|    14050.0|
|          UPI|    41037.5|
+-------------+-----------+



50. Find products generating revenue above 100000.

In [56]:
spark.sql("""
SELECT product_name,
SUM(bill_amount) AS revenue
FROM final_table
GROUP BY product_name
HAVING revenue > 100000
""").show()

+---------------+-------+
|   product_name|revenue|
+---------------+-------+
|         Laptop| 186000|
|Washing Machine| 116000|
|   Mobile Phone| 125000|
+---------------+-------+



Part 6 — Window Functions

51. Rank products by revenue within category.



In [57]:
final_df.groupBy(
    "category",
    "product_name"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "rank",
    rank().over(
        Window.partitionBy("category")
        .orderBy(desc("revenue"))
    )
).show()

+---------------+---------------+-------+----+
|       category|   product_name|revenue|rank|
+---------------+---------------+-------+----+
|          DAIRY|   Cheese Block|   8100|   1|
|          DAIRY|      Milk Pack|   3000|   2|
|    ELECTRONICS|         Laptop| 186000|   1|
|    ELECTRONICS|   Mobile Phone| 125000|   2|
|    ELECTRONICS|Washing Machine| 116000|   3|
|    ELECTRONICS|         LED TV|  90000|   4|
|    ELECTRONICS|Air Conditioner|  84000|   5|
|    ELECTRONICS|   Refrigerator|  76000|   6|
|      GROCERIES|       Rice Bag|  42000|   1|
|      GROCERIES|    Wheat Flour|  31500|   2|
|HOME APPLIANCES| Water Purifier|  48000|   1|
|HOME APPLIANCES|    Ceiling Fan|  33600|   2|
|HOME APPLIANCES|      Gas Stove|  33000|   3|
|HOME APPLIANCES|  Mixer Grinder|  24500|   4|
|  PERSONAL CARE|        Shampoo|   8000|   1|
|  PERSONAL CARE|           Soap|   7200|   2|
|  PERSONAL CARE|     Toothpaste|   3600|   3|
|     STATIONERY|       Pen Pack|  13200|   1|
|     STATION

52. Rank suppliers by revenue within city.


In [58]:
final_df.groupBy(
    "supplier_city",
    "supplier_name"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "rank",
    rank().over(
        Window.partitionBy("supplier_city")
        .orderBy(desc("revenue"))
    )
).show()

+-------------+------------------+-------+----+
|supplier_city|     supplier_name|revenue|rank|
+-------------+------------------+-------+----+
|    Ahmedabad|  National Grocers|  18000|   1|
|    Bengaluru|        OfficeKart|  19200|   1|
|      Chennai|     Kitchen World|  81000|   1|
|      Chennai|   Fresh Dairy Ltd|  11100|   2|
|        Delhi| Elite Electronics| 392000|   1|
|    Hyderabad|     Reddy Traders|  55500|   1|
|    Hyderabad|  Daily Essentials|   8000|   2|
|      Kolkata| Smart Electronics| 285000|   1|
|       Mumbai|CarePlus Suppliers|  10800|   1|
|         Pune| HomeNeeds Pvt Ltd|  58100|   1|
+-------------+------------------+-------+----+



53. Use ROW_NUMBER to find top product per category.


In [59]:
final_df.withColumn(
    "row_num",
    row_number().over(
        Window.partitionBy("category")
        .orderBy(desc("bill_amount"))
    )
).filter(
    col("row_num") == 1
).show()

+--------+-----------+----------+----------+--------+------------+--------------+--------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+-------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|  product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|row_num|
+--------+-----------+----------+----------+--------+------------+--------------+--------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+-------+
|     307|        202|       105|2024-04-04|      18|   Cancelled| 

54. Use DENSE_RANK to rank suppliers by bill amount.


In [60]:
final_df.withColumn(
    "dense_rank",
    dense_rank().over(
        Window.orderBy(desc("bill_amount"))
    )
).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+----------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|dense_rank|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+----------+
|     305|        204|       115|2024-04-03|       3

55. Find top 2 suppliers by revenue.


In [61]:
final_df.groupBy(
    "supplier_name"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "rank",
    rank().over(
        Window.orderBy(desc("revenue"))
    )
).filter(
    col("rank") <= 2
).show()

+-----------------+-------+----+
|    supplier_name|revenue|rank|
+-----------------+-------+----+
|Elite Electronics| 392000|   1|
|Smart Electronics| 285000|   2|
+-----------------+-------+----+



56. Create running total revenue by order date.


In [62]:
final_df.withColumn(
    "running_total",
    sum("bill_amount").over(
        Window.orderBy("order_date")
    )
).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+-------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|running_total|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+-------------+
|     301|        201|       101|2024-04-01

57. Create running total revenue by supplier.


In [63]:
final_df.withColumn(
    "running_total",
    sum("bill_amount").over(
        Window.partitionBy("supplier_name")
        .orderBy("order_date")
    )
).show()

+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+-------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority|   product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|running_total|
+--------+-----------+----------+----------+--------+------------+--------------+---------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+-------------+
|     316|        203|       106|2024-04-08

58. Rank cities by revenue.


In [64]:
final_df.groupBy(
    "supplier_city"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "city_rank",
    rank().over(
        Window.orderBy(desc("revenue"))
    )
).show()

+-------------+-------+---------+
|supplier_city|revenue|city_rank|
+-------------+-------+---------+
|        Delhi| 392000|        1|
|      Kolkata| 285000|        2|
|      Chennai|  92100|        3|
|    Hyderabad|  63500|        4|
|         Pune|  58100|        5|
|    Bengaluru|  19200|        6|
|    Ahmedabad|  18000|        7|
|       Mumbai|  10800|        8|
+-------------+-------+---------+



59. Rank categories by revenue.


In [65]:
final_df.groupBy(
    "category"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "category_rank",
    rank().over(
        Window.orderBy(desc("revenue"))
    )
).show()

+---------------+-------+-------------+
|       category|revenue|category_rank|
+---------------+-------+-------------+
|    ELECTRONICS| 677000|            1|
|HOME APPLIANCES| 139100|            2|
|      GROCERIES|  73500|            3|
|     STATIONERY|  19200|            4|
|  PERSONAL CARE|  18800|            5|
|          DAIRY|  11100|            6|
+---------------+-------+-------------+



60. Find highest bill order per payment mode.

In [66]:
final_df.withColumn(
    "row_num",
    row_number().over(
        Window.partitionBy("payment_mode")
        .orderBy(desc("bill_amount"))
    )
).filter(
    col("row_num") == 1
).show()

+--------+-----------+----------+----------+--------+------------+--------------+-------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+-------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|order_priority| product_name|       category|inventory_city|price|stock_quantity|expensive_product_flag|inventory_value|low_stock_flag|    supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|row_num|
+--------+-----------+----------+----------+--------+------------+--------------+-------------+---------------+--------------+-----+--------------+----------------------+---------------+--------------+-----------------+-------------+---------------+----------+-----------+-------------+--------------+-------+
|     305|        204|       115|2024-04-03|       3|   Delivered|    